# Multi-Horizon Stock Market Forecasting System
## Part 1: Data Pipeline, Feature Engineering & Statistical Baselines

Welcome to the interactive exploration notebook! In this section, we will walk through the core foundations of our forecasting framework:
1. **Data Ingestion Pipeline**: Safely fetching, caching, and cleaning NSE stock data using `yfinance`.
2. **Feature Engineering**: Implementing premium technical indicators (RSI, MACD, Bollinger Bands, Lags, etc.) from scratch in pure Pandas/Numpy.
3. **Stationarity Analysis**: Performing Augmented Dickey-Fuller (ADF) tests to verify stationarity.
4. **Classical Baseline Models**: Fitting Auto-ARIMA and Holt-Winters Exponential Smoothing to set the performance benchmarks.

### 1. Setup & Environment Init

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add source directory to path to import local modules
sys.path.append(os.path.abspath('..'))

from src.utils import logger, setup_plotting_style, plot_predictions
from src.data_pipeline import get_or_fetch_data
from src.feature_engineering import generate_and_save_features, run_stationarity_analysis
from src.models.baselines import BaselineModelWrapper
from src.evaluation import calculate_forecasting_metrics

# Setup plot styles
setup_plotting_style()
print("Environment ready!")

### 2. Ingest Stock Data
We will download historical daily data for `RELIANCE.NS` (Reliance Industries on NSE) and cache it locally inside the `data/raw/` directory.

In [ ]:
ticker = 'RELIANCE.NS'
df_raw = get_or_fetch_data(ticker, start_date='2018-01-01')

print(f"\nDownloaded {len(df_raw)} records.")
print(f"Columns: {df_raw.columns.tolist()}")
print(df_raw.head())

Let's visualize the Closing Price and Volume of our focus stock.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Closing Price
axes[0].plot(df_raw.index, df_raw['Close'], label='Close Price (INR)', color='#2B2D42', linewidth=1.5)
axes[0].set_title(f"{ticker} - Closing Stock Price (NSE)", fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (INR)')
axes[0].legend()

# Volume
axes[1].bar(df_raw.index, df_raw['Volume'], label='Trading Volume', color='#F77F00', alpha=0.6)
axes[1].set_title("Daily Trading Volume", fontsize=13)
axes[1].set_ylabel('Volume')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3. Feature Engineering
We process the raw data to add: Returns, Simple and Exponential Moving Averages, RSI, MACD, Bollinger Bands, Lags, and Calendar features. The resulting feature dataset is cached under `data/processed/`.

In [ ]:
df_features = generate_and_save_features(ticker, df_raw)
print(f"\nFeature engineered dataframe shape: {df_features.shape}")
print(f"Features generated:\n{df_features.columns.tolist()[:25]}... (and more)")

Let's visualize a few premium technical indicators (RSI and MACD) that we engineered.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Prices with BBands
axes[0].plot(df_features.index, df_features['Close'], label='Close', color='black', linewidth=1.2)
axes[0].plot(df_features.index, df_features['BB_Upper'], label='BB Upper', color='#D90429', linestyle=':', alpha=0.8)
axes[0].plot(df_features.index, df_features['BB_Middle'], label='BB Middle (SMA 20)', color='#F77F00', linestyle='--', alpha=0.8)
axes[0].plot(df_features.index, df_features['BB_Lower'], label='BB Lower', color='#00B4D8', linestyle=':', alpha=0.8)
axes[0].set_title(f"{ticker} Close Price with Bollinger Bands", fontsize=13, fontweight='bold')
axes[0].legend(loc='best')

# RSI
axes[1].plot(df_features.index, df_features['RSI'], label='RSI (14)', color='#7209B7', linewidth=1.5)
axes[1].axhline(70, color='#D90429', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(30, color='#00B4D8', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].set_title("Relative Strength Index (RSI)", fontsize=12)
axes[1].set_ylim(10, 90)
axes[1].legend(loc='best')

# MACD
axes[2].plot(df_features.index, df_features['MACD'], label='MACD Line', color='#00B4D8', linewidth=1.2)
axes[2].plot(df_features.index, df_features['MACD_Signal'], label='Signal Line', color='#7209B7', linewidth=1.2)
axes[2].bar(df_features.index, df_features['MACD_Histogram'], label='Histogram', color='gray', alpha=0.4)
axes[2].set_title("Moving Average Convergence Divergence (MACD)", fontsize=12)
axes[2].legend(loc='best')

plt.tight_layout()
plt.show()

### 4. Stationarity Verification
Traditional forecasting models (like ARIMA) require stationary time series. We run the ADF test to examine Close price vs Returns.

In [ ]:
print("=== Stationarity ADF Checks ===")
is_close_stationary = run_stationarity_analysis(df_features['Close'], "Close Price")
is_returns_stationary = run_stationarity_analysis(df_features['Returns'], "Daily Returns")

### 5. Train-Test Split (Chronological)
We use a chronological split (80% train, 20% test) to prevent sequence leakage.

In [ ]:
split_idx = int(len(df_features) * 0.8)
train_data = df_features.iloc[:split_idx]
test_data = df_features.iloc[split_idx:]

print(f"Training set samples: {len(train_data)} ({train_data.index[0].strftime('%Y-%m-%d')} to {train_data.index[-1].strftime('%Y-%m-%d')})")
print(f"Test set samples:     {len(test_data)} ({test_data.index[0].strftime('%Y-%m-%d')} to {test_data.index[-1].strftime('%Y-%m-%d')})")

### 6. Fit Baseline Statistical Models

We will fit **Auto-ARIMA** and **Exponential Smoothing** on the training data and forecast values matching the test set duration.

In [ ]:
steps = len(test_data)

# ARIMA
arima_model = BaselineModelWrapper(model_type='arima')
arima_model.fit(train_data['Close'])
arima_preds = arima_model.predict(steps=steps)

# Exponential Smoothing
expsmooth_model = BaselineModelWrapper(model_type='expsmoothing')
expsmooth_model.fit(train_data['Close'])
expsmooth_preds = expsmooth_model.predict(steps=steps)

### 7. Evaluate Baseline Forecasts

In [ ]:
y_true = test_data['Close'].values

arima_metrics = calculate_forecasting_metrics(y_true, arima_preds, "ARIMA")
expsmooth_metrics = calculate_forecasting_metrics(y_true, expsmooth_preds, "Exponential Smoothing")

# Visual comparison
plt.figure(figsize=(14, 6))
plt.plot(test_data.index, y_true, label='Actual Price', color='#2B2D42', linewidth=2.0)
plt.plot(test_data.index, arima_preds, label='ARIMA Forecast', color='#D90429', linestyle='--')
plt.plot(test_data.index, expsmooth_preds, label='Exp Smoothing Forecast', color='#F77F00', linestyle='--')
plt.title(f"{ticker} - Statistical Forecasts Comparison on Test Data", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.show()

In the next notebook (**Part 2: Deep Learning Models**), we will train custom neural networks in PyTorch (LSTM, GRU, Temporal Attention, and custom Time-Series Transformers) to improve accuracy and perform robust backtesting!